# Bitemporal Change Detection with GeoAI

GeoAI's `train_change_detection()` trains a Siamese or early-fusion network on pairs of bitemporal images to produce a binary change mask. Common use cases: building construction/demolition, deforestation, flood extent before/after.

## References
- GeoAI docs: https://geoai.gishub.org/
- Open-CD (specialized change detection): see `demo/open_cd/` for more advanced methods

## 1. Setup and Data

In [ ]:
import os
import urllib.request
import geoai
import rasterio
from rasterio.plot import show
import matplotlib.pyplot as plt
import numpy as np

# Download bitemporal imagery pair + change labels
os.makedirs("change_data", exist_ok=True)

t1_url, t2_url, label_url = geoai.data.get_change_sample_urls()

t1_path = "change_data/t1.tif"      # before
t2_path = "change_data/t2.tif"      # after
change_path = "change_data/change.tif"  # binary change mask (0=no change, 1=changed)

for url, path in [(t1_url, t1_path), (t2_url, t2_path), (label_url, change_path)]:
    if not os.path.exists(path):
        urllib.request.urlretrieve(url, path)

print("Bitemporal dataset downloaded.")

## 2. Visualize the Temporal Pair

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

with rasterio.open(t1_path) as src:
    show(src, ax=axes[0], title="T1 (before)")

with rasterio.open(t2_path) as src:
    show(src, ax=axes[1], title="T2 (after)")

with rasterio.open(change_path) as src:
    change_arr = src.read(1)
    axes[2].imshow(change_arr, cmap="RdYlGn_r")
    axes[2].set_title(f"Change label ({np.sum(change_arr > 0):,} changed pixels)")
    axes[2].axis("off")

plt.tight_layout()
plt.show()

## 3. Generate Change Detection Chips

In [ ]:
from geoai import chip_change_dataset

chip_change_dataset(
    image_t1=t1_path,
    image_t2=t2_path,
    labels=change_path,
    output_dir="change_chips",
    chip_size=256,
    stride=128,
    min_change_ratio=0.02,  # discard chips with <2% changed pixels
)

chips = [f for f in os.listdir("change_chips/t1") if f.endswith(".tif")]
print(f"Generated {len(chips)} chip triplets (t1/t2/label)")

## 4. Train Change Detection Model

In [ ]:
from geoai import train_change_detection

model, metrics = train_change_detection(
    t1_dir="change_chips/t1",
    t2_dir="change_chips/t2",
    label_dir="change_chips/labels",
    architecture="SiameseUnet",  # or EarlyFusionUnet
    encoder_name="resnet18",
    encoder_weights="imagenet",
    max_epochs=30,
    batch_size=8,
    output_dir="change_output",
    accelerator="auto",
)

print(f"Best val F1: {metrics.get('best_val_f1', 'N/A'):.4f}")

## 5. Full-Scene Inference and Export

In [ ]:
from geoai import predict_change_detection

predict_change_detection(
    model=model,
    image_t1=t1_path,
    image_t2=t2_path,
    output="change_prediction.tif",
    chip_size=256,
    overlap=64,
    threshold=0.5,
)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

with rasterio.open(t1_path) as src:
    show(src, ax=axes[0], title="T1 (before)")

with rasterio.open(t2_path) as src:
    show(src, ax=axes[1], title="T2 (after)")

with rasterio.open("change_prediction.tif") as src:
    pred = src.read(1)
    axes[2].imshow(pred, cmap="RdYlGn_r")
    axes[2].set_title(f"Predicted changes ({np.sum(pred > 0):,} px)")
    axes[2].axis("off")

plt.tight_layout()
plt.show()

## Note: For Advanced Change Detection

GeoAI's change detection uses standard Siamese U-Net or early-fusion approaches. For state-of-the-art methods like ChangeFormer, BAN, or TTP, see the `demo/open_cd/` directory which uses the specialized Open-CD library built on OpenMMLab.